## Document loading

In [11]:
from langchain_community.document_loaders.json_loader import JSONLoader
from langchain_text_splitters import CharacterTextSplitter, RecursiveCharacterTextSplitter
from langchain_community.embeddings import HuggingFaceEmbeddings
# from langchain_nomic import NomicEmbeddings
# from langchain_core.vectorstores import InMemoryVectorStore
from langchain_chroma import Chroma
import numpy as np
from langchain_core.prompts import PromptTemplate
import os
from langchain_huggingface import ChatHuggingFace, HuggingFaceEndpoint
from langchain.messages import HumanMessage,SystemMessage

In [3]:
json = JSONLoader("mini_docs.jsonl", ".text", json_lines=True)
documents = json.load()

## Document chunking

In [4]:
text_splitter = CharacterTextSplitter.from_tiktoken_encoder(
    encoding_name="cl100k_base", chunk_size=100, chunk_overlap=0
)

docs = text_splitter.split_documents(documents)
docs[0]

Document(metadata={'source': '/home/louay/rag/mini_docs.jsonl', 'seq_num': 1}, page_content='langchain_experimental API Reference¶\nlangchain_experimental.agents¶\nFunctions¶\nagents.agent_toolkits.csv.base.create_csv_agent(...)\nCreate csv agent by loading to a dataframe and using pandas agent.\nagents.agent_toolkits.pandas.base.create_pandas_dataframe_agent(llm,\xa0df)\nConstruct a pandas agent from an LLM and dataframe.\nagents.agent_toolkits.python.base.create_python_agent(...)\nConstruct a python agent from an LLM and tool.\nagents.agent_toolkits.spark.base.create_spark_dataframe_agent(llm,\xa0df)\nConstruct a Spark agent from an LLM and dataframe.\nagents.agent_toolkits.xorbits.base.create_xorbits_agent(...)\nConstruct a xorbits agent from an LLM and dataframe.\nlangchain_experimental.autonomous_agents¶\nClasses¶\nautonomous_agents.autogpt.agent.AutoGPT(...)\nAgent class for interacting with Auto-GPT.\nautonomous_agents.autogpt.memory.AutoGPTMemory\nMemory for AutoGPT.\nautonomou

In [ ]:
# emb_dim = 64
embedder = HuggingFaceEmbeddings(
    model_name="sentence-transformers/all-MiniLM-L6-v2"
) 
#NomicEmbeddings(model="nomic-embed-text-v1", inference_mode="local", device="cuda")
# embedded_docs = embedder.embed_documents(texts)
# np.array(embedded_docs[0])

/tmp/ipykernel_3155/120073364.py:2: LangChainDeprecationWarning: The class `HuggingFaceEmbeddings` was deprecated in LangChain 0.2.2 and will be removed in 1.0. An updated version of the class exists in the `langchain-huggingface package and should be used instead. To use it run `pip install -U `langchain-huggingface` and import as `from `langchain_huggingface import HuggingFaceEmbeddings``.
  embedder = HuggingFaceEmbeddings(
/home/louay/miniconda3/envs/rag/lib/python3.13/site-packages/torch/cuda/__init__.py:180: UserWarning: CUDA initialization: The NVIDIA driver on your system is too old (found version 12050). Please update your GPU driver by downloading and installing a new version from the URL: http://www.nvidia.com/Download/index.aspx Alternatively, go to: https://pytorch.org to install a PyTorch version that has been compiled with your version of the CUDA driver. (Triggered internally at /pytorch/c10/cuda/CUDAFunctions.cpp:119.)
  return torch._C._cuda_getDeviceCount() > 0


## Vector storage

In [6]:
store = Chroma.from_documents(
    documents=docs,
    embedding=embedder, 
    persist_directory="./chroma_db"
)
# store.add_documents(docs)

## Query

In [7]:
query = "Give me the langchain_experimental API Reference."
retriever = store.as_retriever(search_type = "similarity", search_kwargs = {"k": 1})
context = retriever.invoke(query)
context

[Document(id='257a244e-f8f4-4cac-bf76-17870c65ba50', metadata={'source': '/home/louay/rag/mini_docs.jsonl', 'seq_num': 1}, page_content='langchain_experimental API Reference¶\nlangchain_experimental.agents¶\nFunctions¶\nagents.agent_toolkits.csv.base.create_csv_agent(...)\nCreate csv agent by loading to a dataframe and using pandas agent.\nagents.agent_toolkits.pandas.base.create_pandas_dataframe_agent(llm,\xa0df)\nConstruct a pandas agent from an LLM and dataframe.\nagents.agent_toolkits.python.base.create_python_agent(...)\nConstruct a python agent from an LLM and tool.\nagents.agent_toolkits.spark.base.create_spark_dataframe_agent(llm,\xa0df)\nConstruct a Spark agent from an LLM and dataframe.\nagents.agent_toolkits.xorbits.base.create_xorbits_agent(...)\nConstruct a xorbits agent from an LLM and dataframe.\nlangchain_experimental.autonomous_agents¶\nClasses¶\nautonomous_agents.autogpt.agent.AutoGPT(...)\nAgent class for interacting with Auto-GPT.\nautonomous_agents.autogpt.memory.A

## Generation

In [12]:
# HF_TOKEN = os.getenv("HUGGINGFACEHUB_API_TOKEN")
SYSTEM_PROMPT = """You are a helpful assistant that answers user queries based primarily on the context information provided to you.
This contexts contains factual information that pertain to the subject. If the context is insufficient to answer, state your inability to answer."""

In [14]:
llm = HuggingFaceEndpoint(
    repo_id="deepseek-ai/DeepSeek-R1-0528",
    task="text-generation",
    max_new_tokens=512,
    do_sample=False,
    repetition_penalty=1.03,
    provider="auto",  # let Hugging Face choose the best provider for you
)

chat_model = ChatHuggingFace(llm=llm)

In [13]:
prompt_template = "The user asked: {user_query} Use the following information to answer accurately:\n{context}."
template = PromptTemplate.from_template(prompt_template)
formatted_prompt = template.format(user_query = query, context = "\n".join([document.page_content for document in context]))

In [15]:
messages = [SystemMessage(SYSTEM_PROMPT), HumanMessage(formatted_prompt)]

In [17]:
ai_msg = chat_model.invoke(messages)
ai_msg.content

"<think>\nWe are given a context that lists various components of the `langchain_experimental` API.\nThe user asks for the API reference, which we have in the context.\n\nWe note that the context is structured in sections:\n\n1. `langchain_experimental.agents` - which contains functions for creating agents for different data sources (CSV, pandas, Spark, etc.)\n2. `langchain_experimental.autonomous_agents` - which contains classes for autonomous agents like AutoGPT and BabyAGI.\n\nWe are to present the information accurately as per the context.\n\nLet's break down the context:\n\n## For `langchain_experimental.agents`:\n  - **Functions**:\n    - `create_csv_agent(...)`: Create csv agent by loading to a dataframe and using pandas agent.\n    - `create_pandas_dataframe_agent(llm, df)`: Construct a pandas agent from an LLM and dataframe.\n    - `create_python_agent(...)`: Construct a python agent from an LLM and tool.\n    - `create_spark_dataframe_agent(llm, df)`: Construct a Spark agent 